# S3, temps 1 : entraînement du projecteur (pilote de 5 000 segments)

Chaîne : vecteurs Cohere Transcribe Arabic **précalculés** (gelé, 12,5 Hz, `d_enc` = 1280) → **projecteur** (LayerNorm + MLP, seul entraîné) → Qwen2.5-1.5B-Instruct **gelé** → transcription arabe.

Ce notebook est une interface fine : toute la logique est dans `code/projecteur/entrainer_temps1.py` et les réglages dans `configs/s3.yaml` (section `temps_1_pilote`).

1. balayage court du taux d'apprentissage ;
2. entraînement (reprise automatique s'il a été coupé) ;
3. courbes de perte et de WER ;
4. évaluation sur le test (épisodes jamais vus) et exemples.

**Ne pas committer ce notebook avec ses sorties** : elles contiennent du texte MGB-2 (licence de recherche QCRI).
Mode essai (petits modèles factices, sans GPU) : variable d'environnement `S3_NOTEBOOK_ESSAI=1`.

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

racine = Path.cwd()
while not (racine / "configs" / "s3.yaml").exists() and racine.parent != racine:
    racine = racine.parent
for dossier in ("projecteur", "ecoute"):
    if str(racine / "code" / dossier) not in sys.path:
        sys.path.insert(0, str(racine / "code" / dossier))

import matplotlib
if os.environ.get("S3_NOTEBOOK_ESSAI") == "1":
    matplotlib.use("Agg")
from courbes_s3 import tracer_balayage, tracer_courbes
from entrainer_temps1 import (balayer_taux, charger_config, chemin_projet, construire_modele, entrainer,
                              evaluer, lire_metriques, preparer_essai)

ESSAI = os.environ.get("S3_NOTEBOOK_ESSAI") == "1"
SECTION = "temps_1_pilote"
config = charger_config(racine / "configs" / "s3.yaml", SECTION)
modele = None
if ESSAI:
    modele = preparer_essai(config, SECTION, Path(tempfile.mkdtemp(prefix="notebook_s3_essai_")))
reglages = config[SECTION]
dossier = chemin_projet(reglages["sortie"])
dossier.mkdir(parents=True, exist_ok=True)
print("sorties :", dossier)
print({k: reglages[k] for k in ("precision_llm", "learning_rate", "taille_lot", "micro_lot", "nb_epoques", "warmup_steps")})

## 1. Balayage du taux d'apprentissage

Quatre entraînements courts (120 pas de 16 segments), même initialisation du projecteur, comparés sur la perte de validation. Les résultats déjà calculés sont relus ; mettre `RELANCER_BALAYAGE = True` pour recommencer.

In [ ]:
RELANCER_BALAYAGE = False
chemin_balayage = dossier / "balayage" / "resultats.json"
if ESSAI or RELANCER_BALAYAGE or not chemin_balayage.is_file():
    valeurs, etapes = ([1e-4, 1e-2], 3) if ESSAI else ([1e-4, 3e-4, 1e-3, 3e-3], 120)
    balayage = balayer_taux(config, valeurs, etapes, SECTION, modele)
    if not ESSAI:
        chemin_balayage.write_text(json.dumps(balayage, indent=2), encoding="utf-8")
else:
    balayage = json.loads(chemin_balayage.read_text(encoding="utf-8"))
for r in balayage:
    print(f"lr {r['learning_rate']:g} : perte de validation {r['perte_validation']:.4f}")
tracer_balayage(balayage, dossier / "balayage.png");

## 2. Entraînement

Lot effectif de 16 (micro-lots de 4, accumulation), lots de longueurs voisines, montée linéaire puis cosinus, gradients écrêtés à 1. Mesure toutes les `eval_toutes_les_etapes` : perte sur 200 segments de validation et WER sur 100 d'entre eux. Le meilleur projecteur (plus petit WER) est gardé dans `projecteur_meilleur.pt`.

Si un entraînement a été coupé, la cellule reprend au dernier point de reprise ; s'il est terminé, elle ne refait rien. Durée mesurée : voir `code/projecteur/journal_s3.md`.

In [ ]:
resultat = entrainer(config, SECTION, modele)
{k: v for k, v in resultat.items() if k != "indices_validation"}

## 3. Courbes

In [ ]:
metriques = lire_metriques(dossier)
tracer_courbes(metriques, dossier / "courbes_temps1.png", titre="S3 temps 1, pilote 5 000 segments : projecteur seul");
validations = [l for l in metriques if l["type"] == "validation"]
for l in validations:
    print(f"étape {l['etape']:5d} (époque {l['epoque']:.2f}) : perte {l['perte']:.3f}"
          + (f", WER {l['wer_pct']:.1f} %, CER {l['cer_pct']:.1f} %" if "wer_pct" in l else ""))
vitesses = [l["segments_par_s"] for l in metriques if l["type"] == "train"]
if vitesses:
    print(f"vitesse d'entraînement : {vitesses[-1]:.1f} segments/s")

## 4. Évaluation sur le test

1 000 segments d'épisodes jamais vus, meilleur projecteur, décodage glouton. WER et CER après `normaliser_arabe` (`code/mesures/scores.py`). Repère : le modèle Cohere complet obtient 15,5 % de WER sur MGB-2 (classement Open Universal Arabic ASR, NON vérifié ici).

In [ ]:
RELANCER_EVALUATION = False
chemin_evaluation = dossier / "evaluation_test.json"
if ESSAI or RELANCER_EVALUATION or not chemin_evaluation.is_file():
    evaluation = evaluer(config, SECTION, modele)
else:
    evaluation = json.loads(chemin_evaluation.read_text(encoding="utf-8"))
evaluation

In [ ]:
# Quelques exemples : référence puis transcription du système
with open(dossier / "hypotheses_test.jsonl", encoding="utf-8") as fichier:
    exemples = [json.loads(ligne) for ligne in fichier]
for exemple in exemples[:: max(1, len(exemples) // 6)][:6]:
    print("RÉF :", exemple["reference"])
    print("S3  :", exemple["hypothese"])
    print()

## 5. Tester un segment

Écouter un segment du test et comparer la référence à la transcription du meilleur projecteur.

In [ ]:
import random
import torch
from donnees_s3 import DatasetS3

jeu_test = DatasetS3(chemin_projet(reglages["dossier_vecteurs"]) / "test")
if modele is None:
    modele = construire_modele(config, reglages, jeu_test.d_enc, torch.device("mps" if torch.backends.mps.is_available() else "cpu"))
modele.projecteur.load_state_dict(torch.load(dossier / "projecteur_meilleur.pt", map_location=modele.device))
modele.eval()
i = random.Random(7).randrange(len(jeu_test))
vecteurs, reference = jeu_test[i]
print("segment :", jeu_test.index[i]["id"], f"({jeu_test.index[i]['duree_s']} s, {vecteurs.shape[0]} vecteurs)")
print("RÉF :", reference)
print("S3  :", modele.generer([vecteurs], "transcription", max_nouveaux_tokens=reglages["max_nouveaux_tokens"],
                           repetition_penalty=reglages["repetition_penalty"])[0])
if not ESSAI:
    from IPython.display import Audio
    from audio_s3 import charger_extrait
    import yaml
    racine_mgb2 = chemin_projet(yaml.safe_load((racine / "configs" / "encodage_s3.yaml").read_text(encoding="utf-8"))["sources"]["racine_mgb2"])
    display(Audio(charger_extrait(jeu_test.index[i], racine_mgb2), rate=16000))